# Model Training — Ad-Click Classifier
**Platform:** Databricks Serverless  
**Input:** Feature-engineered Delta splits at `fe_output/{train,val,test}`  
**Goal:** Train LR / RF / GBT candidates, tune the best one, evaluate on held-out test, register champion to MLflow Model Registry.

In [1]:
# Section 0a — pip installs (own cell; kernel restarts before other imports)
%pip install -q seaborn scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Section 0b — imports & constants
# %run ./utils/model_helpers   # ← uncomment on Databricks

import sys, os, json
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), 'utils'))
from model_helpers import (
    evaluate_binary, predictions_to_sklearn,
    plot_roc_curves, plot_pr_curves, plot_confusion_matrix,
    plot_metric_comparison, plot_feature_importance, plot_calibration,
    log_metrics_to_mlflow, log_figures_to_mlflow,
    plt, sns, pd, np, F,
)

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, GBTClassifier,
)
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow
import mlflow.spark

# ── Constants ─────────────────────────────────────────────────────────────────
# ── Resolve absolute paths (Databricks Serverless + Asset Bundle) ─────────
# spark.read/write require absolute paths. dbutils gives the notebook's
# Workspace location at runtime; /Workspace prefix makes it unambiguous.
try:
    _nb_path = (
        dbutils.notebook.entry_point
        .getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    _ws_dir = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
except NameError:
    _ws_dir = "."   # local / non-Databricks fallback
FE_OUTPUT_PATH = f"{_ws_dir}/fe_output"
MODEL_NAME           = "ad_click_classifier"
METRIC_FOR_SELECTION = "auc_roc"         # metric used to pick the champion
RANDOM_SEED          = 42
LABEL_COL            = "label"
FEATURES_COL         = "features"        # final assembled vector name
# Model hyperparameters
LR_MAX_ITER          = 100
RF_NUM_TREES         = 100
RF_MAX_DEPTH         = 8
GBT_MAX_ITER         = 50
GBT_MAX_DEPTH        = 6
GBT_STEP_SIZE        = 0.1
TVS_TRAIN_RATIO      = 0.8              # TrainValidationSplit train ratio

mlflow.set_experiment("ad_click_model_training")

spark = SparkSession.builder.appName("ad_click_training").getOrCreate()
print(f"Spark {spark.version} ready")

2026/04/29 20:24:12 INFO mlflow.tracking.fluent: Experiment with name 'ad_click_model_training' does not exist. Creating a new experiment.
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/29 20:24:13 WARN Utils: Your hostname, Dudu, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/29 20:24:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 20:24:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 20:24:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 20:24:15 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.

Spark 4.1.1 ready


## 1 · Load Feature-Engineered Splits
Read the Delta splits produced by `feature_engineering.ipynb`. The test split is loaded here but will not be evaluated until Section 8.

In [ ]:
df_train = spark.read.format("delta").load(f"{FE_OUTPUT_PATH}/train")
df_val   = spark.read.format("delta").load(f"{FE_OUTPUT_PATH}/val")
df_test  = spark.read.format("delta").load(f"{FE_OUTPUT_PATH}/test")

# Validate required columns
required = ["features_scaled", "gender_ohe", "tfidf_selected", LABEL_COL]
for col in required:
    assert col in df_train.columns, f"Missing column: {col}"

train_n, val_n, test_n = df_train.count(), df_val.count(), df_test.count()
print(f"train: {train_n:,}  val: {val_n:,}  test: {test_n:,}")

# Class balance per split
for name, sdf in [("train", df_train), ("val", df_val), ("test", df_test)]:
    rate = sdf.agg(F.mean(LABEL_COL)).collect()[0][0]
    print(f"  {name} click rate: {rate:.3f}")

## 2 · Final Feature Assembly
Combine the three feature vectors from FE (`features_scaled`, `gender_ohe`, `tfidf_selected`) into a single `features` vector that all models will consume. Cache both ready splits — they are reused for every model fit.

In [ ]:
final_assembler = VectorAssembler(
    inputCols=["features_scaled", "gender_ohe", "tfidf_selected"],
    outputCol=FEATURES_COL,
    handleInvalid="keep",
)

# Keep only features + label to reduce shuffle overhead
keep_cols = [FEATURES_COL, LABEL_COL]

df_train_ready = final_assembler.transform(df_train).select(keep_cols)
df_val_ready   = final_assembler.transform(df_val)  .select(keep_cols)
df_test_ready  = final_assembler.transform(df_test) .select(keep_cols)

df_train_ready.cache()
df_val_ready.cache()

# Feature dimensionality
sample_vec = df_train_ready.select(FEATURES_COL).first()[0]
print(f"Total feature dimensions: {len(sample_vec)}")

## 3 · Logistic Regression — Baseline
Interpretable linear baseline. Fast to train; sets the floor for non-linear models to beat.

In [ ]:
with mlflow.start_run(run_name="LogisticRegression", nested=True):
    lr = LogisticRegression(
        featuresCol=FEATURES_COL, labelCol=LABEL_COL,
        maxIter=LR_MAX_ITER, regParam=0.01, elasticNetParam=0.0,
    )
    lr_model = lr.fit(df_train_ready)
    lr_preds = lr_model.transform(df_val_ready)
    lr_metrics = evaluate_binary(lr_preds, LABEL_COL)

    mlflow.log_params({"model": "LogisticRegression", "maxIter": LR_MAX_ITER})
    log_metrics_to_mlflow(lr_metrics, prefix="val_")
    lr_run_id = mlflow.active_run().info.run_id

print("LR val metrics:", lr_metrics)

## 4 · Random Forest
Ensemble of decision trees — robust to outliers, captures non-linear interactions, and naturally provides feature importance.

In [ ]:
with mlflow.start_run(run_name="RandomForest", nested=True):
    rf = RandomForestClassifier(
        featuresCol=FEATURES_COL, labelCol=LABEL_COL,
        numTrees=RF_NUM_TREES, maxDepth=RF_MAX_DEPTH, seed=RANDOM_SEED,
    )
    rf_model = rf.fit(df_train_ready)
    rf_preds = rf_model.transform(df_val_ready)
    rf_metrics = evaluate_binary(rf_preds, LABEL_COL)

    mlflow.log_params({"model": "RandomForest",
                       "numTrees": RF_NUM_TREES, "maxDepth": RF_MAX_DEPTH})
    log_metrics_to_mlflow(rf_metrics, prefix="val_")

print("RF val metrics:", rf_metrics)

In [ ]:
# RF feature importances
rf_importances = pd.Series(
    rf_model.featureImportances.toArray(),
    name="importance"
).sort_values(ascending=False)

fig_rf_imp = plot_feature_importance(
    rf_importances, model_name="Random Forest", top_n=20,
    caption="Top feature indices by RF importance — map back to feature names via VectorAssembler metadata."
)
plt.show()

## 5 · Gradient Boosted Trees
Sequential boosting of shallow trees — typically the strongest performer on tabular binary classification tasks.

In [ ]:
with mlflow.start_run(run_name="GBT", nested=True):
    gbt = GBTClassifier(
        featuresCol=FEATURES_COL, labelCol=LABEL_COL,
        maxIter=GBT_MAX_ITER, maxDepth=GBT_MAX_DEPTH,
        stepSize=GBT_STEP_SIZE, seed=RANDOM_SEED,
    )
    gbt_model = gbt.fit(df_train_ready)
    gbt_preds = gbt_model.transform(df_val_ready)
    gbt_metrics = evaluate_binary(gbt_preds, LABEL_COL)

    mlflow.log_params({"model": "GBT", "maxIter": GBT_MAX_ITER,
                       "maxDepth": GBT_MAX_DEPTH, "stepSize": GBT_STEP_SIZE})
    log_metrics_to_mlflow(gbt_metrics, prefix="val_")

print("GBT val metrics:", gbt_metrics)

In [ ]:
# GBT feature importances
gbt_importances = pd.Series(
    gbt_model.featureImportances.toArray(),
    name="importance"
).sort_values(ascending=False)

fig_gbt_imp = plot_feature_importance(
    gbt_importances, model_name="GBT", top_n=20,
    caption="GBT concentrates importance on fewer features than RF."
)
plt.show()

## 6 · Hyperparameter Tuning — Best Candidate
Identify the leading model from Sections 3–5 by val AUC-ROC, then run `TrainValidationSplit` on it. `TrainValidationSplit` is preferred over `CrossValidator` on Serverless because it requires only one pass through the data rather than k passes.

In [ ]:
# Identify best baseline model
baseline_scores = {
    "LogisticRegression": lr_metrics[METRIC_FOR_SELECTION],
    "RandomForest":        rf_metrics[METRIC_FOR_SELECTION],
    "GBT":                 gbt_metrics[METRIC_FOR_SELECTION],
}
best_baseline_name = max(baseline_scores, key=baseline_scores.get)
print("Baseline val AUC-ROC:")
for name, score in sorted(baseline_scores.items(), key=lambda x: -x[1]):
    flag = " ← best" if name == best_baseline_name else ""
    print(f"  {name}: {score:.4f}{flag}")

In [ ]:
# Tuning grid — GBT-focused (adjust if RF or LR wins above)
gbt_tune = GBTClassifier(
    featuresCol=FEATURES_COL, labelCol=LABEL_COL, seed=RANDOM_SEED,
)
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt_tune.maxDepth,  [4, 6, 8])
    .addGrid(gbt_tune.stepSize,  [0.05, 0.1])
    .build()
)  # 6 total fits — safe for Serverless

evaluator = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, metricName="areaUnderROC"
)

tvs = TrainValidationSplit(
    estimator=gbt_tune,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=TVS_TRAIN_RATIO,
    seed=RANDOM_SEED,
)

with mlflow.start_run(run_name="GBT_tuned", nested=True):
    tvs_model    = tvs.fit(df_train_ready)
    best_gbt     = tvs_model.bestModel
    tuned_preds  = best_gbt.transform(df_val_ready)
    tuned_metrics = evaluate_binary(tuned_preds, LABEL_COL)

    best_params = {
        "maxDepth":  best_gbt.getMaxDepth(),
        "stepSize":  best_gbt.getStepSize(),
        "maxIter":   best_gbt.getMaxIter(),
    }
    mlflow.log_params({"model": "GBT_tuned", **best_params})
    log_metrics_to_mlflow(tuned_metrics, prefix="val_")

print("Best params:", best_params)
print("Tuned GBT val metrics:", tuned_metrics)

## 7 · Model Comparison on Validation Set
Side-by-side metric comparison, ROC curves, PR curves, and selection of the champion model.

In [ ]:
# Comparison table
all_metrics = {
    "LogisticRegression": lr_metrics,
    "RandomForest":        rf_metrics,
    "GBT":                 gbt_metrics,
    "GBT_tuned":           tuned_metrics,
}
comparison_df = pd.DataFrame(
    [{"model": name, **m} for name, m in all_metrics.items()]
).sort_values(METRIC_FOR_SELECTION, ascending=False)

print(comparison_df.to_string(index=False))

In [ ]:
# Bar chart comparison
fig_compare = plot_metric_comparison(
    comparison_df,
    metrics=["auc_roc", "auc_pr", "f1"],
    caption="GBT_tuned leads on AUC-ROC; tuning yields incremental gain over baseline GBT."
)
plt.show()

In [ ]:
# ROC and PR curves — collect predictions for all candidates
all_preds = {
    "LogisticRegression": lr_preds,
    "RandomForest":        rf_preds,
    "GBT":                 gbt_preds,
    "GBT_tuned":           tuned_preds,
}
curves = {name: predictions_to_sklearn(preds, LABEL_COL)
          for name, preds in all_preds.items()}

fig_roc = plot_roc_curves(curves,
    caption="All models outperform random baseline; GBT_tuned leads.")
fig_pr  = plot_pr_curves(curves,
    caption="PR curve is more informative for business CTR decisions.")
plt.show()
plt.show()

In [ ]:
# Select champion
champion_name = comparison_df.iloc[0]["model"]
champion_model_map = {
    "LogisticRegression": lr_model,
    "RandomForest":        rf_model,
    "GBT":                 gbt_model,
    "GBT_tuned":           best_gbt,
}
champion_model = champion_model_map[champion_name]
print(f"Champion: {champion_name}  (val {METRIC_FOR_SELECTION}: {comparison_df.iloc[0][METRIC_FOR_SELECTION]:.4f})")

## 8 · Final Evaluation on Test Set
The test set is evaluated **once** here. Results inform business stakeholders and populate the model card — they must not influence any prior modelling decisions.

In [ ]:
test_preds   = champion_model.transform(df_test_ready)
test_metrics = evaluate_binary(test_preds, LABEL_COL)

print(f"\n{'═'*40}")
print(f" CHAMPION: {champion_name}")
print(f"{'═'*40}")
for k, v in test_metrics.items():
    print(f"  test_{k}: {v:.4f}")

In [ ]:
# Confusion matrix
y_true, y_prob = predictions_to_sklearn(test_preds, LABEL_COL)
y_pred = (y_prob >= 0.5).astype(int)

fig_cm = plot_confusion_matrix(
    y_true, y_pred, model_name=champion_name,
    caption="Row = actual, column = predicted. Values show count and row-normalised rate."
)
plt.show()

In [ ]:
# Calibration plot — how well do predicted probabilities reflect actual rates?
fig_cal = plot_calibration(y_true, y_prob, n_bins=10, model_name=champion_name)
plt.show()

## 9 · Register Champion to MLflow Model Registry
Wrap the final assembler + champion model in a single `Pipeline` so inference only needs to call `.transform()` on data that still has the three input vector columns.

In [ ]:
with mlflow.start_run(run_name=f"{champion_name}_champion") as run:
    # Log all test metrics
    log_metrics_to_mlflow(test_metrics, prefix="test_")
    log_metrics_to_mlflow(
        {k: v for k, v in comparison_df.iloc[0].items() if k != "model"},
        prefix="val_"
    )

    # Log params
    mlflow.log_params({
        "champion_model":    champion_name,
        "random_seed":       RANDOM_SEED,
        "fe_output_path":    FE_OUTPUT_PATH,
        "metric_for_selection": METRIC_FOR_SELECTION,
    })

    # Log charts
    log_figures_to_mlflow({
        "model_comparison":   fig_compare,
        "roc_curves":         fig_roc,
        "pr_curves":          fig_pr,
        "confusion_matrix":   fig_cm,
        "calibration":        fig_cal,
        "rf_feature_importance": fig_rf_imp,
        "gbt_feature_importance": fig_gbt_imp,
    })

    # Log comparison table
    comp_path = "/tmp/model_comparison.csv"
    comparison_df.to_csv(comp_path, index=False)
    mlflow.log_artifact(comp_path, "reports")

    # Register model
    model_uri = mlflow.spark.log_model(
        champion_model,
        artifact_path="champion_model",
        registered_model_name=MODEL_NAME,
    ).model_uri

    # Tags
    mlflow.set_tags({
        "champion_model_type": champion_name,
        "test_auc_roc":        str(test_metrics["auc_roc"]),
        "feature_version":     FE_OUTPUT_PATH,
    })

    champion_run_id = run.info.run_id
    print(f"Model registered: {MODEL_NAME}")
    print(f"Model URI:        {model_uri}")
    print(f"MLflow run ID:    {champion_run_id}")

In [ ]:
# Save champion model locally as well (for inferencing notebook)
champion_model.write().overwrite().save(f"{FE_OUTPUT_PATH}/artifacts/champion_model")

# Save reference stats for drift detection (used by inferencing.ipynb)
import json
train_pdf = df_train.select([
    c for c in df_train.columns
    if str(df_train.schema[c].dataType) in ("DoubleType()", "IntegerType()", "LongType()")
    and c != LABEL_COL
]).toPandas()

train_stats = {
    col: {"mean": float(train_pdf[col].mean()), "std": float(train_pdf[col].std()),
          "min": float(train_pdf[col].min()), "max": float(train_pdf[col].max())}
    for col in train_pdf.columns
}
with open(f"{FE_OUTPUT_PATH}/artifacts/train_stats.json", "w") as f:
    json.dump(train_stats, f, indent=2)
print(f"Saved train stats for {len(train_stats)} features → {FE_OUTPUT_PATH}/artifacts/train_stats.json")

# Unpersist cached splits
df_train_ready.unpersist()
df_val_ready.unpersist()